# Data Preprocessing

In this notebook we prepare the data for model training.
We drop irrelevant columns, encode the target variable,
split into train/test sets, and scale the features.

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

import pickle

In [6]:
df = pd.read_csv('../data/raw_data.csv')

df.head()

,age,gender,academic_year,study_hours_per_day,exam_pressure,academic_performance,stress_level,anxiety_score,depression_score,sleep_hours,physical_activity,social_support,screen_time,internet_usage,financial_stress,family_expectation,burnout_score,mental_health_index,risk_level,dropout_risk
0,23,Male,2,5.596071,6.487218,68.411114,4.116950,2.275713,1.986730,6.880545,2.728861,6.470080,4.993801,4.983157,3.446626,3.586147,2.037344,7.074487,Low,1.746601
1,20,Male,3,5.597171,5.631481,67.682159,0.349489,0.000000,0.000000,7.463339,3.690866,0.000000,3.862980,5.136124,2.814039,5.478666,0.000000,9.860204,Low,0.000000
2,29,Male,2,2.580491,6.015297,58.372363,3.476177,2.425201,0.851996,8.946670,3.296720,6.901725,5.428880,3.058333,4.918515,6.068155,0.000000,7.626370,Low,0.696941
3,27,Male,4,4.607208,6.684005,68.925653,6.778843,4.512425,4.285645,4.571380,2.065480,2.349857,6.304842,6.931147,6.915885,6.557540,7.227651,4.649042,High,5.380592
4,24,Male,4,2.186569,4.010945,69.141915,1.854595,1.102558,0.000000,5.989324,4.026504,4.512921,4.903146,5.134903,4.382820,5.934779,0.000000,8.927394,Low,0.000000


### Drop unnecessary columns
We drop columns that cause data leakage (`burnout_score`, `mental_health_index`, `dropout_risk`)
and columns that showed near-zero correlation with `risk_level` in EDA.

In [7]:
df_clean = df.drop(columns=['burnout_score', "mental_health_index", "dropout_risk"])

In [8]:
df_clean = df_clean.drop(columns=["academic_year", "screen_time", "internet_usage", "academic_performance"]) # near zero corelation with our target 


### Note on age and gender

We intentionally keep `age` and `gender` in the feature set even though
they showed near-zero correlation with `risk_level` in our EDA.

The reason is user experience — when a student fills in the survey form,
entering their age and gender makes the interaction feel more personal
and complete. These fields also allow for potential demographic insights
in future versions of the app (e.g. does burnout differ by gender or age group?).

`gender` will be encoded as: Male=0, Female=1, Other=2
`age` is already numeric and will be scaled along with the other features.

In [9]:
# Encode gender
gender_map = {'Male': 0, 'Female': 1, 'Other': 2}
df_clean['gender'] = df_clean['gender'].map(gender_map)

print("Gender encoded:", df_clean['gender'].unique())

Gender encoded: [0 1 2]


### Encode the target variable
`risk_level` is currently text (Low, Medium, High).
We map it to numbers so the model can work with it: Low=0, Medium=1, High=2.

In [10]:
risk_map = {'Low':0,'Medium':1,'High':2}

df_clean['risk_level'] = df_clean['risk_level'].map(risk_map)

print("Unique values after encoding:", df_clean['risk_level'].unique())
print("Value counts:\n", df_clean['risk_level'].value_counts())

Unique values after encoding: [0 2 1]
Value counts:
 risk_level
0    766645
1    218275
2     15080
Name: count, dtype: int64


### Define X and y
X contains all input features. y contains the target (risk_level encoded).

In [11]:
X = df_clean.drop(columns=['risk_level'])
y = df_clean['risk_level']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", X.columns.tolist())

y.head()

X shape: (1000000, 12)
y shape: (1000000,)
Features: ['age', 'gender', 'study_hours_per_day', 'exam_pressure', 'stress_level', 'anxiety_score', 'depression_score', 'sleep_hours', 'physical_activity', 'social_support', 'financial_stress', 'family_expectation']


0    0
1    0
2    0
3    2
4    0
Name: risk_level, dtype: int64

### Train/test split
We split 80% of the data for training and keep 20% for testing.
random_state=42 ensures the split is reproducible.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)
print("y_train distribution:\n", y_train.value_counts())
print("y_test distribution:\n",  y_test.value_counts())

X_train: (800000, 12)
X_test:  (200000, 12)
y_train distribution:
 risk_level
0    613316
1    174620
2     12064
Name: count, dtype: int64
y_test distribution:
 risk_level
0    153329
1     43655
2      3016
Name: count, dtype: int64


### Scale the features
StandardScaler standardises features to have mean=0 and std=1.
We fit ONLY on X_train to avoid data leakage, then transform both sets.

In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("X_train_scaled mean (should be ~0):", X_train_scaled.mean().round(4))
print("X_train_scaled std  (should be ~1):", X_train_scaled.std().round(4))

X_train_scaled mean (should be ~0): 0.0
X_train_scaled std  (should be ~1): 1.0


### Save scaler and processed data
We save the scaler and processed arrays so notebook 03 can load them directly
without re-running preprocessing every time.

In [39]:
import pickle
import numpy as np

# Save scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save processed arrays
np.save('../models/X_train.npy', X_train_scaled)
np.save('../models/X_test.npy',  X_test_scaled)
np.save('../models/y_train.npy', y_train.values)
np.save('../models/y_test.npy',  y_test.values)

# Save feature names — needed later in the backend
with open('../models/feature_names.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print("Saved: scaler.pkl, X_train.npy, X_test.npy, y_train.npy, y_test.npy, feature_names.pkl")

Saved: scaler.pkl, X_train.npy, X_test.npy, y_train.npy, y_test.npy, feature_names.pkl


In [14]:
print(df_clean[['exam_pressure', 'financial_stress', 'family_expectation', 'stress_level']].describe())


        exam_pressure  financial_stress  family_expectation    stress_level
count  1000000.000000    1000000.000000      1000000.000000  1000000.000000
mean         5.998810          5.002920            5.983287        4.246462
std          1.548268          1.976426            1.960972        1.678998
min          1.000000          0.000000            0.000000        0.000000
25%          4.944647          3.656767            4.647065        3.102593
50%          5.998906          5.001279            5.999517        4.244029
75%          7.051914          6.354571            7.351578        5.385464
max         10.000000         10.000000           10.000000       10.000000
